# QuantJourney SDK - Consumer Dispersion Not The Trade

This notebook demonstrates a QuantJourney SDK workflow that separates consumer exposure into discretionary, staples, retail, travel and balance-sheet-quality buckets using macro data, sector ETFs, fundamentals and prices.

It covers:

- Direct QuantJourney SDK calls for the required market, macro, regulatory or portfolio data
- Transparent pandas/numpy calculations so research assumptions stay visible
- Chart-ready output that can be reused in notebooks, reports or API documentation

## Prerequisites

Make sure you have:

- Access to QuantJourney API (https://api.quantjourney.cloud)
- `QJ_API_KEY` configured in your environment
- Tenant access to the connectors used by this example

## Imports and Plot Style

In [ ]:
import os
import math
import json
from pathlib import Path
from typing import Any
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from quantjourney.sdk import QuantJourneyAPI
plt.style.use('default')
plt.rcParams.update({'figure.figsize': (12, 6), 'axes.grid': True, 'grid.alpha': 0.25, 'axes.spines.top': False, 'axes.spines.right': False})


## QuantJourney Client

In [ ]:
qj = QuantJourneyAPI(api_key=os.environ['QJ_API_KEY'])
START = os.getenv('QJ_EXAMPLE_START', '2020-01-01')
END = os.getenv('QJ_EXAMPLE_END') or pd.Timestamp.today().normalize().strftime('%Y-%m-%d')


## Response Helpers

In [ ]:
def unwrap(payload: Any) -> Any:
    """Return the useful data value from common QuantJourney response shapes."""
    if isinstance(payload, dict) and 'data' in payload:
        payload = payload['data']
    if isinstance(payload, dict) and 'value' in payload:
        return payload['value']
    return payload

def as_rows(payload: Any) -> list[dict[str, Any]]:
    value = unwrap(payload)
    if value is None:
        return []
    if isinstance(value, list):
        return value
    if isinstance(value, dict):
        for key in ('rows', 'data', 'items', 'prices', 'results'):
            if isinstance(value.get(key), list):
                return value[key]
        return [value]
    return []


## Market Data Helpers

In [ ]:
def price_frame(symbol: str, start: str=START, end: str=END) -> pd.DataFrame:
    payload = qj.eod.get_historical_prices(symbol=symbol, start_date=start, end_date=end)
    rows = as_rows(payload)
    if not rows and isinstance(unwrap(payload), dict):
        value = unwrap(payload)
        rows = value.get(symbol) or value.get(symbol.upper()) or []
    df = pd.DataFrame(rows)
    if df.empty:
        raise RuntimeError(f'No price data returned for {symbol}')
    df['date'] = pd.to_datetime(df['date'])
    for col in ['open', 'high', 'low', 'close', 'adjusted_close', 'volume']:
        if col in df:
            df[col] = pd.to_numeric(df[col], errors='coerce')
    if 'adjusted_close' in df and df['adjusted_close'].notna().any():
        df['price'] = df['adjusted_close'].fillna(df['close'])
    else:
        df['price'] = df['close']
    if 'volume' not in df:
        df['volume'] = np.nan
    return df.dropna(subset=['price']).sort_values('date').set_index('date')

def price_panel(symbols: list[str], start: str=START, end: str=END) -> tuple[pd.DataFrame, pd.DataFrame]:
    prices = {}
    volumes = {}
    for symbol in symbols:
        df = price_frame(symbol, start=start, end=end)
        prices[symbol] = df['price']
        volumes[symbol] = df['volume']
    return (pd.DataFrame(prices).dropna(how='all'), pd.DataFrame(volumes).reindex(pd.DataFrame(prices).index))

def returns(prices: pd.DataFrame) -> pd.DataFrame:
    return prices.pct_change().replace([np.inf, -np.inf], np.nan).dropna(how='all')

def dollar_adv(prices: pd.DataFrame, volumes: pd.DataFrame, window: int=63) -> pd.DataFrame:
    return (prices * volumes).rolling(window).mean()


In [ ]:
symbols = ['XLY', 'XLP', 'WMT', 'COST', 'HD', 'LOW', 'MCD', 'SBUX', 'TSLA', 'AMZN', 'NKE']
macro_raw = {'Retail Sales (FRED:RSAFS)': qj.fred.get_fred_data_series_by_id(series_id='RSAFS', start='2018-01-01'), 'Consumer Sentiment (FRED:UMCSENT)': qj.fred.get_fred_data_series_by_id(series_id='UMCSENT', start='2018-01-01'), 'Consumer Credit (FRED:TOTALSL)': qj.fred.get_fred_data_series_by_id(series_id='TOTALSL', start='2018-01-01'), 'CPI Urban Consumers (FRED:CPIAUCSL)': qj.fred.get_cpi(start_date='2018-01-01'), 'Effective Fed Funds Rate (FRED:FEDFUNDS)': qj.fred.get_effective_federal_funds_rate(start_date='2018-01-01')}
ratios_raw = {symbol: qj.fmp.get_financial_ratios_ttm(symbol=symbol) for symbol in symbols[2:]}
profiles_raw = {symbol: qj.fmp.get_company_profile(symbol=symbol) for symbol in symbols[2:]}
prices, volumes = price_panel(symbols, start='2018-01-01', end=END)


In [ ]:
def first_row(payload: Any) -> dict[str, Any]:
    rows = as_rows(payload)
    return rows[0] if rows and isinstance(rows[0], dict) else {}
ret = returns(prices)
rows = []
for symbol in symbols[2:]:
    ratios = first_row(ratios_raw[symbol])
    profile = first_row(profiles_raw[symbol])
    rows.append({'symbol': symbol, 'industry': profile.get('industry') or profile.get('sector'), 'return_126d': prices[symbol].pct_change(126).iloc[-1], 'volatility_63d': ret[symbol].tail(63).std() * np.sqrt(252), 'gross_margin_ttm': pd.to_numeric(ratios.get('grossProfitMarginTTM'), errors='coerce'), 'current_ratio_ttm': pd.to_numeric(ratios.get('currentRatioTTM'), errors='coerce'), 'debt_to_equity_ttm': pd.to_numeric(ratios.get('debtEquityRatioTTM'), errors='coerce')})
consumer = pd.DataFrame(rows).set_index('symbol')


In [ ]:
consumer['resilience_score'] = consumer['gross_margin_ttm'].rank(pct=True) + consumer['current_ratio_ttm'].rank(pct=True) - consumer['debt_to_equity_ttm'].rank(pct=True) + consumer['return_126d'].rank(pct=True) - consumer['volatility_63d'].rank(pct=True)
etf_spread = prices['XLP'].pct_change(126) - prices['XLY'].pct_change(126)
macro_rows = pd.Series({name: len(as_rows(payload)) for name, payload in macro_raw.items()})
display(macro_rows.rename('macro_rows'))
display(consumer.sort_values('resilience_score', ascending=False))
consumer[['return_126d', 'gross_margin_ttm', 'current_ratio_ttm', 'debt_to_equity_ttm', 'resilience_score']].plot(kind='bar', subplots=True, layout=(2, 3), figsize=(15, 8), title='Consumer dispersion inputs')
plt.tight_layout()
plt.show()


## Notes

This is an example workflow. In production, tenant scopes, connector allowlists,
provider metadata, request IDs and audit logs should be retained next to the resulting
tables or charts.